# PointsX - LV-MHP-v2 Dataset Exploration

Visualize the LV-MHP-v2 dataset: pose annotations, parsing masks, and dataset statistics.

In [ ]:
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image
from scipy.io import loadmat

sns.set_theme(style="whitegrid")

DATA_ROOT = Path("../data/LV-MHP-v2")
COLORMAP = loadmat(str(DATA_ROOT / "LV-MHP-v2_colormap.mat"))["MHP_colormap"]

## 1. Dataset Overview

In [ ]:
splits = {}
for split in ["train", "val"]:
    img_dir = DATA_ROOT / split / "images"
    pose_dir = DATA_ROOT / split / "pose_annos"
    parse_dir = DATA_ROOT / split / "parsing_annos"
    splits[split] = {
        "images": sorted(img_dir.glob("*.jpg")),
        "poses": sorted(pose_dir.glob("*.mat")),
        "parsing": sorted(parse_dir.glob("*.png")),
    }

test_imgs = sorted((DATA_ROOT / "test" / "images").glob("*.jpg"))

print(f"{'Split':<8} {'Images':>8} {'Pose .mat':>10} {'Parsing .png':>13}")
print("-" * 42)
for split, d in splits.items():
    print(f"{split:<8} {len(d['images']):>8} {len(d['poses']):>10} {len(d['parsing']):>13}")
print(f"{'test':<8} {len(test_imgs):>8} {'—':>10} {'—':>13}")
print("-" * 42)
total_imgs = sum(len(d["images"]) for d in splits.values()) + len(test_imgs)
print(f"{'Total':<8} {total_imgs:>8}")

## 2. Image Size Distribution

In [ ]:
# Sample images for size stats (full scan is slow)
rng = np.random.default_rng(42)
sample_paths = rng.choice(splits["train"]["images"], size=min(2000, len(splits["train"]["images"])), replace=False)

widths, heights = [], []
for p in sample_paths:
    img = Image.open(p)
    widths.append(img.width)
    heights.append(img.height)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(widths, bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Image Width Distribution")
axes[0].set_xlabel("Width (px)")

axes[1].hist(heights, bins=40, color="coral", edgecolor="white")
axes[1].set_title("Image Height Distribution")
axes[1].set_xlabel("Height (px)")

axes[2].scatter(widths, heights, alpha=0.3, s=8, color="teal")
axes[2].set_title("Width vs Height")
axes[2].set_xlabel("Width (px)")
axes[2].set_ylabel("Height (px)")
axes[2].set_aspect("equal")

fig.suptitle(f"Image Dimensions (n={len(sample_paths)} train samples)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Width  — min: {min(widths)}, max: {max(widths)}, median: {int(np.median(widths))}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, median: {int(np.median(heights))}")

## 3. Persons-per-Image Distribution

In [ ]:
persons_per_image = {}
for split in ["train", "val"]:
    counts = []
    for mat_path in splits[split]["poses"]:
        data = loadmat(str(mat_path))
        n_persons = sum(1 for k in data if k.startswith("person_"))
        counts.append(n_persons)
    persons_per_image[split] = counts

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, split in zip(axes, ["train", "val"]):
    counts = persons_per_image[split]
    max_val = max(counts)
    bins = np.arange(0.5, max_val + 1.5, 1)
    ax.hist(counts, bins=bins, color="steelblue" if split == "train" else "coral", edgecolor="white", rwidth=0.85)
    ax.set_title(f"{split.capitalize()} — Persons per Image")
    ax.set_xlabel("Number of persons")
    ax.set_ylabel("Number of images")
    ax.set_xticks(range(1, min(max_val + 1, 20)))
    mean_val = np.mean(counts)
    ax.axvline(mean_val, color="red", linestyle="--", linewidth=1.5, label=f"Mean: {mean_val:.1f}")
    ax.legend()

plt.tight_layout()
plt.show()

for split, counts in persons_per_image.items():
    print(f"{split}: min={min(counts)}, max={max(counts)}, mean={np.mean(counts):.2f}, total_persons={sum(counts)}")

## 4. Keypoint Definitions

| Index | Keypoint | Index | Keypoint |
|-------|----------|-------|----------|
| 0 | Right ankle | 8 | Upper neck |
| 1 | Right knee | 9 | Head top |
| 2 | Right hip | 10 | Right wrist |
| 3 | Left hip | 11 | Right elbow |
| 4 | Left knee | 12 | Right shoulder |
| 5 | Left ankle | 13 | Left shoulder |
| 6 | Pelvis | 14 | Left elbow |
| 7 | Thorax | 15 | Left wrist |

## 5. Keypoint Visibility Statistics

In [ ]:
KP_NAMES = [
    "R-Ankle", "R-Knee", "R-Hip", "L-Hip", "L-Knee", "L-Ankle",
    "Pelvis", "Thorax", "Neck", "Head",
    "R-Wrist", "R-Elbow", "R-Shoulder", "L-Shoulder", "L-Elbow", "L-Wrist",
]
VIS_LABELS = {0: "Visible", 1: "Occluded", 2: "Not labeled"}

# Count visibility per keypoint (sample for speed)
vis_counts = np.zeros((16, 3), dtype=int)  # [kp_idx, vis_value]

sample_mats = rng.choice(splits["train"]["poses"], size=min(3000, len(splits["train"]["poses"])), replace=False)
total_persons_sampled = 0

for mat_path in sample_mats:
    data = loadmat(str(mat_path))
    for key in data:
        if not key.startswith("person_"):
            continue
        kps = data[key]  # (20, 3)
        total_persons_sampled += 1
        for i in range(16):
            v = int(kps[i, 2])
            if 0 <= v <= 2:
                vis_counts[i, v] += 1

# Stacked bar chart
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(16)
colors = ["#2ecc71", "#f39c12", "#e74c3c"]

bottom = np.zeros(16)
for v_idx, (v_label, color) in enumerate(zip(["Visible", "Occluded", "Not labeled"], colors)):
    pcts = vis_counts[:, v_idx] / vis_counts.sum(axis=1) * 100
    ax.bar(x, pcts, bottom=bottom, label=v_label, color=color, edgecolor="white", width=0.7)
    bottom += pcts

ax.set_xticks(x)
ax.set_xticklabels(KP_NAMES, rotation=45, ha="right")
ax.set_ylabel("Percentage")
ax.set_title(f"Keypoint Visibility Distribution (n={total_persons_sampled} persons)")
ax.legend(loc="upper right")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## 6. Keypoint Spatial Heatmap

In [ ]:
# Collect normalized keypoint positions
all_kps_norm = []  # list of (x_norm, y_norm) per keypoint

for mat_path in sample_mats:
    stem = mat_path.stem
    img_path = DATA_ROOT / "train" / "images" / f"{stem}.jpg"
    if not img_path.exists():
        continue
    img = Image.open(img_path)
    w, h = img.size

    data = loadmat(str(mat_path))
    for key in data:
        if not key.startswith("person_"):
            continue
        kps = data[key]
        for i in range(16):
            x, y, v = kps[i]
            if v == 0 and x >= 0 and y >= 0:  # visible
                all_kps_norm.append((x / w, y / h, i))

kps_arr = np.array(all_kps_norm)
print(f"Total visible keypoints collected: {len(kps_arr)}")

# Overall heatmap
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# First plot: all keypoints
ax = axes[0]
ax.hist2d(kps_arr[:, 0], kps_arr[:, 1], bins=50, cmap="hot")
ax.set_title("All Keypoints")
ax.set_xlim(0, 1)
ax.set_ylim(1, 0)  # flip y
ax.set_aspect("equal")

# Per-group heatmaps
groups = {
    "Legs": [0, 1, 2, 3, 4, 5],
    "Torso": [6, 7, 8, 9],
    "Right Arm": [10, 11, 12],
    "Left Arm": [13, 14, 15],
    "Head+Neck": [8, 9],
    "Hips": [2, 3, 6],
    "Shoulders": [12, 13],
}

for ax, (name, indices) in zip(axes[1:], groups.items()):
    mask = np.isin(kps_arr[:, 2].astype(int), indices)
    subset = kps_arr[mask]
    if len(subset) > 0:
        ax.hist2d(subset[:, 0], subset[:, 1], bins=50, cmap="hot")
    ax.set_title(name)
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    ax.set_aspect("equal")

fig.suptitle("Keypoint Spatial Distribution (normalized coords)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Bounding Box Size Distribution

In [ ]:
bbox_widths, bbox_heights, bbox_areas_rel = [], [], []

for mat_path in sample_mats:
    stem = mat_path.stem
    img_path = DATA_ROOT / "train" / "images" / f"{stem}.jpg"
    if not img_path.exists():
        continue
    img = Image.open(img_path)
    iw, ih = img.size

    data = loadmat(str(mat_path))
    for key in data:
        if not key.startswith("person_"):
            continue
        kps = data[key]
        x1, y1 = kps[18, 0], kps[18, 1]  # bbox top-left
        x2, y2 = kps[19, 0], kps[19, 1]  # bbox bottom-right
        if x1 < 0 or y1 < 0 or x2 < 0 or y2 < 0:
            continue
        bw = (x2 - x1) / iw
        bh = (y2 - y1) / ih
        if bw > 0 and bh > 0:
            bbox_widths.append(bw)
            bbox_heights.append(bh)
            bbox_areas_rel.append(bw * bh)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(bbox_widths, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Bbox Width (normalized)")
axes[0].set_xlabel("Width / Image Width")

axes[1].hist(bbox_heights, bins=50, color="coral", edgecolor="white")
axes[1].set_title("Bbox Height (normalized)")
axes[1].set_xlabel("Height / Image Height")

axes[2].hist(bbox_areas_rel, bins=50, color="teal", edgecolor="white")
axes[2].set_title("Bbox Area (normalized)")
axes[2].set_xlabel("(W*H) / (ImgW*ImgH)")

fig.suptitle(f"Person Bounding Box Distributions (n={len(bbox_widths)})", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Bbox width  — median: {np.median(bbox_widths):.3f}, mean: {np.mean(bbox_widths):.3f}")
print(f"Bbox height — median: {np.median(bbox_heights):.3f}, mean: {np.mean(bbox_heights):.3f}")
print(f"Bbox area   — median: {np.median(bbox_areas_rel):.3f}, mean: {np.mean(bbox_areas_rel):.3f}")

## 8. Parsing (Segmentation) Class Distribution

In [ ]:
PARSE_CLASSES = [
    "Background", "Cap/hat", "Helmet", "Face", "Hair",
    "Left-arm", "Right-arm", "Left-hand", "Right-hand", "Protector",
    "Bikini/bra", "Jacket/hoodie", "Tee-shirt", "Polo-shirt", "Sweater",
    "Singlet", "Torso-skin", "Pants", "Shorts", "Skirt",
    "Stockings", "Socks", "L-boot", "R-boot", "L-shoe",
    "R-shoe", "L-highheel", "R-highheel", "L-sandal", "R-sandal",
    "L-leg", "R-leg", "L-foot", "R-foot", "Coat",
    "Dress", "Robe", "Jumpsuit", "Full-body", "Headwear",
    "Backpack", "Ball", "Bats", "Belt", "Bottle",
    "Carrybag", "Cases", "Sunglasses", "Eyewear", "Glove",
    "Scarf", "Umbrella", "Wallet/purse", "Watch", "Wristband",
    "Tie", "Other-acc", "Other-upper", "Other-lower",
]

# Sample parsing annotations
sample_parse = rng.choice(splits["train"]["parsing"], size=min(3000, len(splits["train"]["parsing"])), replace=False)

pixel_counts = np.zeros(59, dtype=np.int64)
for p in sample_parse:
    mask = np.array(Image.open(p))
    if mask.ndim == 3:
        mask = mask[:, :, 0]  # take first channel if RGBA
    vals, cnts = np.unique(mask, return_counts=True)
    for v, c in zip(vals, cnts):
        if v < 59:
            pixel_counts[v] += c

# Exclude background for visibility
fg_counts = pixel_counts[1:]
fg_names = PARSE_CLASSES[1:]

# Sort by frequency
order = np.argsort(fg_counts)[::-1]
top_n = 30

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(
    range(top_n),
    fg_counts[order[:top_n]],
    color=plt.cm.viridis(np.linspace(0.2, 0.8, top_n)),
    edgecolor="white",
)
ax.set_yticks(range(top_n))
ax.set_yticklabels([fg_names[i] for i in order[:top_n]])
ax.invert_yaxis()
ax.set_xlabel("Pixel Count")
ax.set_title(f"Top {top_n} Parsing Classes by Pixel Count (n={len(sample_parse)} masks)")
plt.tight_layout()
plt.show()

## 9. Visualize Samples — Pose Annotations

In [ ]:
SKELETON = [
    (0, 1), (1, 2),       # right leg
    (3, 4), (4, 5),       # left leg
    (2, 6), (3, 6),       # hips to pelvis
    (6, 7),               # pelvis to thorax
    (7, 8), (8, 9),       # thorax to head
    (7, 12), (12, 11), (11, 10),  # right arm
    (7, 13), (13, 14), (14, 15),  # left arm
]

LIMB_COLORS = [
    "#e6194b", "#e6194b",             # right leg (red)
    "#3cb44b", "#3cb44b",             # left leg (green)
    "#ffe119", "#ffe119",             # hips (yellow)
    "#4363d8",                         # torso (blue)
    "#f58231", "#f58231",             # neck/head (orange)
    "#911eb4", "#911eb4", "#911eb4",  # right arm (purple)
    "#42d4f4", "#42d4f4", "#42d4f4",  # left arm (cyan)
]


def draw_pose(ax, img, mat_data):
    """Draw pose skeleton on image."""
    ax.imshow(img)
    person_keys = sorted(k for k in mat_data if k.startswith("person_"))

    for person_key in person_keys:
        kps = mat_data[person_key]  # (20, 3)

        # Draw skeleton
        for (i, j), color in zip(SKELETON, LIMB_COLORS):
            if kps[i, 2] == 0 and kps[j, 2] == 0 and kps[i, 0] >= 0 and kps[j, 0] >= 0:
                ax.plot([kps[i, 0], kps[j, 0]], [kps[i, 1], kps[j, 1]],
                        color=color, linewidth=2, alpha=0.8)

        # Draw keypoints
        for i in range(16):
            x, y, v = kps[i]
            if v == 0 and x >= 0:  # visible
                ax.plot(x, y, "o", color="white", markersize=5, markeredgecolor="black", markeredgewidth=1)
            elif v == 1 and x >= 0:  # occluded
                ax.plot(x, y, "s", color="yellow", markersize=4, markeredgecolor="black", markeredgewidth=1, alpha=0.6)

        # Draw bbox
        x1, y1 = kps[18, 0], kps[18, 1]
        x2, y2 = kps[19, 0], kps[19, 1]
        if x1 >= 0 and y1 >= 0 and x2 >= 0 and y2 >= 0:
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       linewidth=1.5, edgecolor="lime", facecolor="none", linestyle="--")
            ax.add_patch(rect)

    ax.axis("off")


# Visualize random samples
sample_indices = rng.choice(len(splits["train"]["poses"]), size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, idx in zip(axes.flatten(), sample_indices):
    mat_path = splits["train"]["poses"][idx]
    img_path = DATA_ROOT / "train" / "images" / f"{mat_path.stem}.jpg"
    img = np.array(Image.open(img_path))
    mat_data = loadmat(str(mat_path))
    n_persons = sum(1 for k in mat_data if k.startswith("person_"))
    draw_pose(ax, img, mat_data)
    ax.set_title(f"{mat_path.stem}.jpg ({n_persons} persons)", fontsize=10)

fig.suptitle("Pose Annotation Samples", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Visualize Samples — Parsing (Segmentation) Masks

In [ ]:
def get_parsing_overlay(img_path, parse_dir):
    """Combine all person parsing masks into one overlay."""
    img = np.array(Image.open(img_path))
    stem = img_path.stem
    mask_paths = sorted(parse_dir.glob(f"{stem}_*.png"))

    combined = np.zeros(img.shape[:2], dtype=np.uint8)
    for mp in mask_paths:
        m = np.array(Image.open(mp))
        if m.ndim == 3:
            m = m[:, :, 0]
        combined = np.where(m > 0, m, combined)

    # Colorize
    color_mask = np.zeros((*combined.shape, 3), dtype=np.uint8)
    for cls_id in range(1, 59):
        color_mask[combined == cls_id] = (COLORMAP[cls_id] * 255).astype(np.uint8)

    # Blend
    overlay = (img * 0.5 + color_mask * 0.5).astype(np.uint8)
    overlay[combined == 0] = img[combined == 0]

    return img, overlay, len(mask_paths)


sample_imgs = rng.choice(splits["train"]["images"], size=4, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, img_path in enumerate(sample_imgs):
    img, overlay, n_masks = get_parsing_overlay(img_path, DATA_ROOT / "train" / "parsing_annos")
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"{img_path.stem}.jpg", fontsize=10)
    axes[0, i].axis("off")
    axes[1, i].imshow(overlay)
    axes[1, i].set_title(f"Parsing ({n_masks} masks)", fontsize=10)
    axes[1, i].axis("off")

fig.suptitle("Parsing Annotation Samples (original + segmentation overlay)", fontsize=14)
plt.tight_layout()
plt.show()

## 11. Combined View — Pose + Parsing

In [ ]:
sample_imgs = rng.choice(splits["train"]["images"], size=4, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
for i, img_path in enumerate(sample_imgs):
    stem = img_path.stem

    # Original
    img = np.array(Image.open(img_path))
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"{stem}.jpg", fontsize=10)
    axes[0, i].axis("off")

    # Pose
    mat_path = DATA_ROOT / "train" / "pose_annos" / f"{stem}.mat"
    if mat_path.exists():
        mat_data = loadmat(str(mat_path))
        draw_pose(axes[1, i], img, mat_data)
        n_p = sum(1 for k in mat_data if k.startswith("person_"))
        axes[1, i].set_title(f"Pose ({n_p} persons)", fontsize=10)
    else:
        axes[1, i].imshow(img)
        axes[1, i].axis("off")

    # Parsing
    _, overlay, n_masks = get_parsing_overlay(img_path, DATA_ROOT / "train" / "parsing_annos")
    axes[2, i].imshow(overlay)
    axes[2, i].set_title(f"Parsing ({n_masks} masks)", fontsize=10)
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12, rotation=0, labelpad=60, va="center")
axes[1, 0].set_ylabel("Pose", fontsize=12, rotation=0, labelpad=60, va="center")
axes[2, 0].set_ylabel("Parsing", fontsize=12, rotation=0, labelpad=60, va="center")

fig.suptitle("Combined View: Original / Pose / Parsing", fontsize=14)
plt.tight_layout()
plt.show()